# Midterm 1: run the data generator in Colab

Use this notebook if you cannot run Python or git on your own machine. It only **generates and shows your data**. You write your answers in `ANSWERS.md` on **your fork on github.com** (open the file, click the pencil icon, edit, *Commit changes*), then submit the commit URL in Canvas. See the README for the rules.

1. Fill in `GITHUB_USERNAME` (the account that owns your fork) and `CODE` (`PRACTICE` to rehearse; the real code is on the board during the exam).
2. Run the three code cells in order.

In [2]:
GITHUB_USERNAME = "bekarystolkymbai"
CODE = "MAPLE-66"
assert GITHUB_USERNAME, "fill in GITHUB_USERNAME first"

In [3]:
import os

if not os.path.exists("midterm-1-AI-course"):
    !git clone --quiet https://github.com/unreal-kz/midterm-1-AI-course.git
%cd midterm-1-AI-course
!python make_my_data.py --github {GITHUB_USERNAME} --code {CODE}

/content/midterm-1-AI-course
Wrote 3 files to my_data/ · fingerprint 5726f126
Next: open my_data/DATA.md


In [4]:
import pandas as pd
from IPython.display import Markdown, display

display(Markdown(open("my_data/DATA.md", encoding="utf-8").read()))
ranks = pd.read_csv("my_data/retrieval_ranks.csv")   # Part 1
docs = pd.read_csv("my_data/doc_results.csv")        # Part 2
display(ranks.head(), docs)

# Midterm 1: your data

GitHub username: `bekarystolkymbai` · session code: `maple-66` · generator v1

**Fingerprint: `5726f126`**: copy it into the first lines of `ANSWERS.md`. All numbers below are synthetic: this is an exam, not a measurement.

## Part 1: inputs

`retrieval_ranks.csv`: 60 test questions (20 EN, 20 RU, 20 KK). `rank_A` and `rank_B` give the position of the correct chunk in the top 10 retrieved under config A and config B; **11 means the correct chunk was not in the top 10.**

Production traffic and Model M token counts (tokens per item, as counted by Model M's tokenizer):

| language | share of traffic | question tokens | answer tokens | chunk A tokens | chunk B tokens |
|---|---|---|---|---|---|
| EN | 18 % | 37 | 106 | 274 | 65 |
| RU | 44 % | 75 | 215 | 556 | 132 |
| KK | 38 % | 85 | 245 | 633 | 150 |

- Requests per day: **9,500**, in the traffic shares above.
- System prompt: **258** tokens in every request.
- Price: **$1** per million input tokens, **$6** per million output tokens.
- Annual budget for this service (model calls only): **$9,300**.

## Part 2: inputs

`doc_results.csv`: for each pipeline, language and scan condition, `n_docs` test invoices and `n_correct` = invoices whose number, date, seller ID and total were all extracted exactly (strict scorer, as in Lab 04).

- **Pipeline H**: hosted vision-language API in a data centre abroad, pay per token; each page is sent as an image.
- **Pipeline L**: open vision-language model on your own server in Kazakhstan, fixed monthly cost.

- Volume: **6,000** scanned pages per day. In **34 %** of them an individual's name and ИИН are printed (sole proprietors, receipts).
- Page image: **1654 × 2339** pixels. Pipeline H counts an image as ⌈width ÷ 28⌉ × ⌈height ÷ 28⌉ input tokens and returns about **365** output tokens per page (the JSON).
- Pipeline H price: **$3** per million input tokens, **$15** per million output tokens.
- Pipeline L: one server costs **$600** per month and handles up to **6,000** pages per day; add servers as needed.

## Part 3: your five questions

Answer each in `ANSWERS.md`: one **Number** line and one **Why** line of at most 40 words.

### Q1

One complaint sentence, measured three ways (synthetic numbers):

| | characters | UTF-8 bytes | tokens (Model M) |
|---|---|---|---|
| EN | 315 | 315 | 58 |
| RU | 321 | 605 | 118 |
| KK | 353 | 686 | 134 |

Compute the KK÷EN ratio for characters, for bytes and for tokens. **Number:** the token ratio divided by the byte ratio (2 decimals). **Why** (at most 40 words): what does this number say about where the Kazakh premium comes from, the encoding or the tokenizer?

### Q2

A teammate sets the chunk size to **576 tokens, counted with cl100k**. The retrieval model (e5-small) reads at most 512 of *its own* tokens. Assume e5 tokens grow in proportion to cl100k tokens at the rate measured in Lecture 4.1 (a chunker told 256 cl100k tokens produced chunks of 263 / 128 / 76 e5 tokens for EN / RU / KK), and that every chunk is exactly the full size.

**Number:** the share of an English chunk, in percent (1 decimal), that the embedding never sees. **Why** (at most 40 words): which of the three languages are cut, and what happens to the cut part?

### Q3

A language model gives these next-token scores for the prompt "The branch is closed on":

| token | score |
|---|---|
| Monday | 4.2 |
| Friday | 3.9 |
| Sunday | 2.9 |
| holiday | 1.9 |
| Easter | 1.5 |

Use temperature T = 1, then apply top-p = 0.9. **Number:** the probability of the top token after top-p and renormalisation, in percent (1 decimal). **Why** (at most 40 words): how does top-p differ from lowering the temperature?

### Q4

Two models answer the same 100 factual questions (1 point per correct answer on the public leaderboard):

| | correct | wrong | "I don't know" |
|---|---|---|---|
| Model X | 74 | 26 | 0 |
| Model Y | 71 | 5 | 24 |

Now score them again with a penalty: +1 per correct answer, −2 per wrong answer, 0 for "I don't know". **Number:** Y's penalised score minus X's (negative if X is ahead). **Why** (at most 40 words): which model does the public leaderboard prefer, and what does that teach a model to do?

### Q5

A support chat adds **86 new tokens per turn** (customer message plus assistant reply) and has **17 turns**. The model keeps no state, so the full history is sent again on every turn (Lecture 1.2). As in the Lecture 1.2 table (100, 200, 300, ... tokens billed on turns 1, 2, 3, ...), turn *i* is billed for the tokens of all *i* turns so far, including its own. **Number:** the total input tokens billed over the whole conversation. **Why** (at most 40 words): how many times has the first message been paid for by the last turn, and what does prompt caching change?


,qid,lang,rank_A,rank_B
0,q01,RU,1,2
1,q02,RU,1,4
2,q03,EN,1,1
3,q04,RU,5,11
4,q05,KK,1,1


,pipeline,lang,condition,n_docs,n_correct
0,H,EN,clean,30,29
1,H,EN,degraded,30,25
2,H,RU,clean,30,28
3,H,RU,degraded,30,25
4,H,KK,clean,30,27
5,H,KK,degraded,30,23
6,L,EN,clean,30,28
7,L,EN,degraded,30,23
8,L,RU,clean,30,27
9,L,RU,degraded,30,22


`ranks` and `docs` are ready for you to compute with. Remember the **fingerprint** at the top of the data sheet: it goes into the first lines of `ANSWERS.md`.

In [8]:
import os
import re
import math
import numpy as np
import pandas as pd

# 1. Загрузка данных и вспомогательные функции
DATA_PATH = "my_data/DATA.md"
RANKS_PATH = "/content/retrieval_ranks.csv"
DOCS_PATH = "/content/doc_results.csv"

assert os.path.exists(DATA_PATH), "Сначала запустите make_my_data.py!"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data_text = f.read()

ranks_df = pd.read_csv(RANKS_PATH)
docs_df = pd.read_csv(DOCS_PATH)

def wilson(hits, n=60, z=1.96):
    p = hits / n
    denom = 1.0 + (z**2) / n
    centre = (p + (z**2) / (2.0 * n)) / denom
    half = (z * math.sqrt((p * (1.0 - p) / n) + ((z**2) / (4.0 * n**2)))) / denom
    return p, max(0.0, centre - half), min(1.0, centre + half)

def softmax(scores, temperature=1.0):
    m = max(scores)
    exps = [math.exp((s - m) / temperature) for s in scores]
    total = sum(exps)
    return [e / total for e in exps]

print("="*60)
print("=== РЕЗУЛЬТАТЫ ДЛЯ ANSWERS.MD ===")
print("="*60)

# PART 1: Retrieval and Cost Decision
print("\n### PART 1: Retrieval Metrics & Candidates\n")

# 1a. Общие метрики для Config A и Config B
for cfg in ['A', 'B']:
    col = f"rank_{cfg}"
    r1 = (ranks_df[col] <= 1).sum() / 60
    r3 = (ranks_df[col] <= 3).sum() / 60
    r5 = (ranks_df[col] <= 5).sum() / 60
    mrr10 = ranks_df[col].apply(lambda r: 1.0 / r if r <= 10 else 0.0).mean()
    print(f"Config {cfg}: Recall@1 = {r1:.3f} | Recall@3 = {r3:.3f} | Recall@5 = {r5:.3f} | MRR@10 = {mrr10:.3f}")

# Parsing parameters from DATA.md for Part 1
system_tokens = int(re.search(r"System prompt: \*\*(\d+)\*\*", data_text).group(1))
req_per_day = int(re.search(r"Requests per day: \*\*([\d,]+)\*\*", data_text).group(1).replace(",", ""))
p_in = float(re.search(r"Price: \*\*\$([\d.]+)\*\*", data_text).group(1))
p_out = float(re.search(r"per million input tokens, \*\*\$([\d.]+)\*\*", data_text).group(1))
budget = int(re.search(r"Annual budget for this service \(model calls only\): \*\*\$([\d,]+)\*\*", data_text).group(1).replace(",", ""))

# Parse language table
shares, q_toks, a_toks, chunk_A_toks, chunk_B_toks = {}, {}, {}, {}, {}
table_match = re.findall(r"\|\s*(EN|RU|KK)\s*\|\s*(\d+)\s*%\s*\|\s*(\d+)\s*\|\s*(\d+)\s*\|\s*(\d+)\s*\|\s*(\d+)\s*\|", data_text)
for lang, sh, q_t, a_t, ca_t, cb_t in table_match:
    shares[lang] = float(sh) / 100.0
    q_toks[lang] = int(q_t)
    a_toks[lang] = int(a_t)
    chunk_A_toks[lang] = int(ca_t)
    chunk_B_toks[lang] = int(cb_t)

chunk_toks = {'A': chunk_A_toks, 'B': chunk_B_toks}

print("\n--- Candidate Details (Part 1b & 1c) ---")
cands_res = []
for cfg, k in [("A", 3), ("A", 5), ("B", 3), ("B", 5)]:
    col = f"rank_{cfg}"
    hits = (ranks_df[col] <= k).sum()
    p_rec, lo, hi = wilson(hits, 60)

    # Cost calculation
    cost_per_req = 0.0
    for lang in ["EN", "RU", "KK"]:
        in_t = system_tokens + q_toks[lang] + k * chunk_toks[cfg][lang]
        out_t = a_toks[lang]
        c = (in_t * p_in + out_t * p_out) / 1e6
        cost_per_req += shares[lang] * c
    ann_cost = cost_per_req * req_per_day * 365

    cands_res.append({"cand": f"({cfg},{k})", "cfg": cfg, "k": k, "hits": hits, "recall": p_rec, "lo": lo, "hi": hi, "cost": ann_cost})
    print(f"Candidate ({cfg},{k}): Hits={hits}/60 | Recall@k={p_rec:.3f} | 95% CI=[{lo:.3f}, {hi:.3f}] | Annual Cost=${ann_cost:,.2f} {'(WITHIN BUDGET)' if ann_cost <= budget else '(EXCEEDS BUDGET)'}")

print(f"\nBudget Limit: ${budget:,}")

print("\n--- Recall по языкам (Part 1d) ---")
for cfg, k in [("A", 3), ("A", 5), ("B", 3), ("B", 5)]:
    col = f"rank_{cfg}"
    sub_recalls = {}
    for lang in ["EN", "RU", "KK"]:
        lang_hits = (ranks_df[ranks_df['lang'] == lang][col] <= k).sum()
        sub_recalls[lang] = lang_hits / 20.0
    print(f"Candidate ({cfg},{k}): EN={sub_recalls['EN']:.2f}, RU={sub_recalls['RU']:.2f}, KK={sub_recalls['KK']:.2f}")


# PART 2: A Document Pipeline You Can Defend
print("\n" + "="*60)
print("### PART 2: Document Pipeline Results\n")

pages_per_day = int(re.search(r"Volume: \*\*([\d,]+)\*\*", data_text).group(1).replace(",", ""))
pii_share = float(re.search(r"In \*\*(\d+)\s*%\*\*", data_text).group(1)) / 100.0
width, height = map(int, re.search(r"Page image: \*\*(\d+)\s*×\s*(\d+)\*\*", data_text).groups())
out_toks_page = int(re.search(r"about \*\*(\d+)\*\*", data_text).group(1))
p_in_doc = float(re.search(r"Pipeline H price: \*\*\$([\d.]+)\*\*", data_text).group(1))
p_out_doc = float(re.search(r"per million input tokens, \*\*\$([\d.]+)\*\*", data_text).group(1))
server_month = float(re.search(r"one server costs \*\*\$([\d,]+)\*\*", data_text).group(1).replace(",", ""))
server_cap = int(re.search(r"handles up to \*\*([\d,]+)\*\*", data_text).group(1).replace(",", ""))

# Annual PII pages
pii_pages_year = pages_per_day * 365 * pii_share

# Pipeline H Cost
img_tokens = math.ceil(width / 28.0) * math.ceil(height / 28.0)
cost_per_page_H = (img_tokens * p_in_doc + out_toks_page * p_out_doc) / 1e6
annual_cost_H = pages_per_day * 365 * cost_per_page_H

# Pipeline L Cost
servers_needed = math.ceil(pages_per_day / server_cap)
annual_cost_L = servers_needed * server_month * 12

print(f"Scanned pages carrying personal data exported via H: {pii_pages_year:,.0f} pages/year\n")

for pipe in ["H", "L"]:
    pipe_df = docs_df[docs_df['pipeline'] == pipe]
    tot_correct = pipe_df['n_correct'].sum()
    tot_docs = pipe_df['n_docs'].sum()
    overall_acc = tot_correct / tot_docs

    pipe_df = pipe_df.copy()
    pipe_df['acc'] = pipe_df['n_correct'] / pipe_df['n_docs']
    worst_row = pipe_df.loc[pipe_df['acc'].idxmin()]
    w_hits = worst_row['n_correct']
    w_n = worst_row['n_docs']
    w_acc, w_lo, w_hi = wilson(w_hits, w_n)

    pipe_cost = annual_cost_H if pipe == "H" else annual_cost_L

    print(f"Pipeline {pipe}:")
    print(f"  Overall Accuracy = {overall_acc:.3%}")
    print(f"  Weakest Cell = ({worst_row['lang']}, {worst_row['condition']}) | Acc = {w_acc:.1%} ({w_hits}/{w_n}) | 95% CI = [{w_lo:.3f}, {w_hi:.3f}]")
    print(f"  Annual Cost = ${pipe_cost:,.2f}\n")


# PART 3: Explain Your Own Numbers
print("="*60)
print("### PART 3: Short Questions Solutions\n")

# Q1
q1_text = re.search(r"### Q1\n\n(.*?)(?=\n### Q2|\Z)", data_text, re.S).group(1)
print("--- Q1 ---")
if "chars" in q1_text or "character" in q1_text:
    chars_en = int(re.search(r"\| EN \| (\d+) \|", q1_text).group(1))
    chars_kk = int(re.search(r"\| KK \| (\d+) \|", q1_text).group(1))
    bytes_en = int(re.search(r"\| EN \| \d+ \| (\d+) \|", q1_text).group(1))
    bytes_kk = int(re.search(r"\| KK \| \d+ \| (\d+) \|", q1_text).group(1))
    toks_en = int(re.search(r"\| EN \| \d+ \| \d+ \| (\d+) \|", q1_text).group(1))
    toks_kk = int(re.search(r"\| KK \| \d+ \| \d+ \| (\d+) \|", q1_text).group(1))

    token_ratio = toks_kk / toks_en
    byte_ratio = bytes_kk / bytes_en
    ans_q1 = token_ratio / byte_ratio
    print(f"Number: {ans_q1:.2f}")
    print("Why: The premium comes from the tokenizer's vocabulary, not UTF-8 encoding; Kazakh text is split into finer, shorter subword fragments than English.")
else:
    in_en = int(re.search(r"\| EN \| ([\d,]+) \|", q1_text).group(1).replace(",", ""))
    out_en = int(re.search(r"\| EN \| [\d,]+ \| ([\d,]+) \|", q1_text).group(1).replace(",", ""))
    in_kk = int(re.search(r"\| KK \| ([\d,]+) \|", q1_text).group(1).replace(",", ""))
    out_kk = int(re.search(r"\| KK \| [\d,]+ \| ([\d,]+) \|", q1_text).group(1).replace(",", ""))
    p_in_q1 = float(re.search(r"\(\$([\d.]+) per million input", q1_text).group(1))
    p_out_q1 = float(re.search(r"per million input tokens, \$([\d.]+) per million output", q1_text).group(1))

    cost_en = (in_en * p_in_q1 + out_en * p_out_q1) / 1e6
    cost_kk = (in_kk * p_in_q1 + out_kk * p_out_q1) / 1e6
    bill_ratio = cost_kk / cost_en
    print(f"Number: {bill_ratio:.2f}")
    print("Why: Input and output tokens have vastly different prices; since input and output token ratios differ, the total bill ratio diverges from the raw token count ratio.")

# Q2
q2_text = re.search(r"### Q2\n\n(.*?)(?=\n### Q3|\Z)", data_text, re.S).group(1)
print("\n--- Q2 ---")
size_q2 = int(re.search(r"chunk size to \*\*(\d+) tokens", q2_text).group(1))
e5_tokens_en = size_q2 * (263.0 / 256.0)

if "share of an English chunk, in percent" in q2_text:
    unseen_tokens = max(0.0, e5_tokens_en - 512.0)
    unseen_pct = (unseen_tokens / e5_tokens_en) * 100.0
    print(f"Number: {unseen_pct:.1f}%")
    print("Why: English is truncated because e5 exceeds 512 tokens. Cut text is ignored during vector embedding, making those facts unsearchable without triggering any runtime errors.")
else:
    frac_q2 = float(re.search(r"starts (\d+) % of the way", q2_text).group(1)) / 100.0
    start_pos = round(e5_tokens_en * frac_q2)
    print(f"Number: {start_pos}")
    print("Why: If the sentence starts beyond token 512, e5 truncates it completely, preventing retrieval for matches in that sentence without throwing any system errors.")

# Q3
q3_text = re.search(r"### Q3\n\n(.*?)(?=\n### Q4|\Z)", data_text, re.S).group(1)
print("\n--- Q3 ---")
logits_match = re.findall(r"\| [A-Za-z]+ \| ([\d\.-]+) \|", q3_text)
logits = [float(x) for x in logits_match]

if "temperature T =" in q3_text:
    temp_q3 = float(re.search(r"temperature T = ([\d\.]+)", q3_text).group(1))
    probs = softmax(logits, temp_q3)
    top_p_pct = probs[0] * 100.0
    print(f"Number: {top_p_pct:.1f}%")
    print("Why: Temperature scales probabilities without altering rank order or token eligibility; lower T sharpens the distribution, increasing the top token's probability.")
else:
    nucleus = float(re.search(r"top-p = ([\d\.]+)", q3_text).group(1))
    probs = softmax(logits, 1.0)
    cum = 0.0
    kept_probs = []
    for pr in probs:
        cum += pr
        kept_probs.append(pr)
        if cum >= nucleus:
            break
    renorm_top_pct = (kept_probs[0] / sum(kept_probs)) * 100.0
    print(f"Number: {renorm_top_pct:.1f}%")
    print("Why: Top-p dynamically truncates the low-probability tail and renormalizes remaining tokens, whereas temperature scales all logits without cutting off the candidate set.")

# Q4
q4_text = re.search(r"### Q4\n\n(.*?)(?=\n### Q5|\Z)", data_text, re.S).group(1)
print("\n--- Q4 ---")
if "Model X" in q4_text:
    c_x = int(re.search(r"\| Model X \| (\d+) \|", q4_text).group(1))
    c_y = int(re.search(r"\| Model Y \| (\d+) \|", q4_text).group(1))
    a_y = int(re.search(r"\| Model Y \| \d+ \| \d+ \| (\d+) \|", q4_text).group(1))
    w_x = 100 - c_x
    w_y = 100 - c_y - a_y
    penalty = int(re.search(r"−(\d+) per wrong answer", q4_text).group(1))

    score_x = c_x - penalty * w_x
    score_y = c_y - penalty * w_y
    diff = score_y - score_x
    print(f"Number: {diff}")
    print("Why: The public leaderboard rewards aggressive guessing over admitting ignorance, teaching models to hallucinate rather than decline answering when uncertain.")
else:
    sets_matches = re.findall(r"\| (.*?) \| (\d+) \| (\d+) % \|", q4_text)
    total_q, bound_q = 0, 0
    for name, n_q, pct in sets_matches:
        n_q, pct = int(n_q), int(pct)
        if "multiplication" in name.lower() or "arithmetic" in name.lower():
            continue
        total_q += n_q
        bound_q += n_q * (pct / 100.0)
    bound_pct = (bound_q / total_q) * 100.0
    print(f"Number: {bound_pct:.1f}%")
    print("Why: Arithmetic sets were excluded because math relies on procedural logic, not single-fact memorization; system prompt instructions cannot override missing pre-training facts.")

# Q5
q5_text = re.search(r"### Q5\n\n(.*?)(?=\n|\Z)", data_text, re.S).group(1)
print("\n--- Q5 ---")
if "billion parameters on" in q5_text:
    params_b = float(re.search(r"\*\*(\d+) billion parameters", q5_text).group(1))
    if "trillion" in q5_text:
        tokens_b = float(re.search(r"on ([\d\.]+) trillion tokens\*\*", q5_text).group(1)) * 1000.0
    else:
        tokens_b = float(re.search(r"on ([\d,]+) billion tokens\*\*", q5_text).group(1).replace(",", ""))

    my_tpp = tokens_b / params_b
    chinchilla_tpp = 1400.0 / 70.0
    ratio_q5 = my_tpp / chinchilla_tpp
    print(f"Number: {ratio_q5:.2f}")
    status = "over-trained" if ratio_q5 > 1.0 else "under-trained"
    print(f"Why: The model is {status} relative to compute-optimal scaling; Chinchilla demonstrated that earlier models like GPT-3 were severely undertrained for their size.")
else:
    per_turn = int(re.search(r"\*\*(\d+) new tokens per turn\*\*", q5_text).group(1))
    turns = int(re.search(r"\*\*(\d+) turns\*\*", q5_text).group(1))

    # Сумма прогрессии: sum_{i=1}^{N} (i * per_turn)
    total_billed = per_turn * (turns * (turns + 1)) // 2
    print(f"Number: {total_billed}")
    print(f"Why: The first message is re-read and paid for {turns} times; prompt caching eliminates this quadratic cost by caching repeated context serverside.")

print("="*60)

=== РЕЗУЛЬТАТЫ ДЛЯ ANSWERS.MD ===

### PART 1: Retrieval Metrics & Candidates

Config A: Recall@1 = 0.717 | Recall@3 = 0.850 | Recall@5 = 0.917 | MRR@10 = 0.787
Config B: Recall@1 = 0.333 | Recall@3 = 0.517 | Recall@5 = 0.800 | MRR@10 = 0.481

--- Candidate Details (Part 1b & 1c) ---
Candidate (A,3): Hits=51/60 | Recall@k=0.850 | 95% CI=[0.739, 0.919] | Annual Cost=$11,006.33 (EXCEEDS BUDGET)
Candidate (A,5): Hits=55/60 | Recall@k=0.917 | 95% CI=[0.819, 0.964] | Annual Cost=$14,713.09 (EXCEEDS BUDGET)
Candidate (B,3): Hits=31/60 | Recall@k=0.517 | 95% CI=[0.393, 0.638] | Annual Cost=$6,765.02 (WITHIN BUDGET)
Candidate (B,5): Hits=48/60 | Recall@k=0.800 | 95% CI=[0.682, 0.882] | Annual Cost=$7,644.24 (WITHIN BUDGET)

Budget Limit: $9,300

--- Recall по языкам (Part 1d) ---
Candidate (A,3): EN=0.95, RU=0.80, KK=0.80
Candidate (A,5): EN=0.95, RU=0.90, KK=0.90
Candidate (B,3): EN=0.60, RU=0.40, KK=0.55
Candidate (B,5): EN=0.85, RU=0.80, KK=0.75

### PART 2: Document Pipeline Results

Scann